# Vamana runs

End-to-end pipeline for one dataset:

1. **Convert** an ANN-benchmarks `.hdf5` into the `.fbin` format ParlayANN reads.
2. **Run Vamana** to build the graph (shells out to ParlayANN's `neighbors` binary).
3. **Compute coverage stats** and write the `(neighbor, uncov)` adj-list.

Step 3 uses the GPU when CuPy is available and falls back to NumPy otherwise.

## Configuration

Everything the run needs is set here; the rest of the notebook reads these.

In [ ]:
from pathlib import Path

import numpy as np

# --- paths ---------------------------------------------------------------
HDF5_PATH   = Path("../../Datasets/fashion_mnist-784-euclidean.hdf5")
DATA_DIR    = Path("../../Datasets/fashion_mnist-784-euclidean")   # fbin output
BUILT_DIR   = Path("../../Built-Graphs")                           # graph + adj-list
VAMANA_BIN  = Path("../ParlayANN/algorithms/vamana/neighbors")

# --- vamana build params -------------------------------------------------
R     = 128   # max out-degree
L     = 128   # beam width during construction (needs L >= R)
ALPHA = 1.0   # prune slack; 1.0 = strict pruning

# --- coverage ------------------------------------------------------------
COVERAGE_ALPHA = 1.0    # alpha the coverage is measured at
LIMIT = None            # cap nodes for a quick check; None = whole graph
DTYPE = "float64"       # float32 halves memory but makes boundary ties unstable

TAG = f"vamana-fashion_mnist-784-euclidean-R{R}-alpha{ALPHA:g}"
GRAPH_PATH = BUILT_DIR / TAG
ADJ_PATH   = BUILT_DIR / f"adj-list-{TAG}.txt"
BASE_FBIN  = DATA_DIR / "base.fbin"
QUERY_FBIN = DATA_DIR / "query.fbin"

for d in (DATA_DIR, BUILT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"graph    -> {GRAPH_PATH}")
print(f"adj-list -> {ADJ_PATH}")

## 1. HDF5 to fbin

An `.fbin` file is a little-endian int32 `n`, an int32 `dims`, then `n * dims` row-major
float32 values. The `train` and `test` datasets are written to separate files.

In [ ]:
import h5py


def write_fbin(vectors, path):
    """Write a 2D array to `path` in fbin format."""
    vectors = np.ascontiguousarray(vectors, dtype=np.float32)
    n, dims = vectors.shape
    with open(path, "wb") as f:
        np.array([n, dims], dtype=np.int32).tofile(f)
        vectors.tofile(f)
    print(f"wrote {path} ({n} points, dimension {dims})")


def read_fbin(path):
    """Read an fbin file back into a 2D array."""
    with open(path, "rb") as f:
        n, dims = np.fromfile(f, dtype=np.int32, count=2)
        return np.fromfile(f, dtype=np.float32).reshape(n, dims)


def hdf5_to_fbin(hdf5_path, base_out, query_out):
    """Convert the train and test datasets into separate fbin files."""
    with h5py.File(hdf5_path, "r") as f:
        write_fbin(f["train"][:], base_out)
        write_fbin(f["test"][:], query_out)

In [ ]:
if BASE_FBIN.exists() and QUERY_FBIN.exists():
    print(f"fbin files already present in {DATA_DIR}, skipping conversion")
else:
    hdf5_to_fbin(HDF5_PATH, BASE_FBIN, QUERY_FBIN)

## 2. Run Vamana

Shells out to ParlayANN's `neighbors`. Build the binary first with `make` in
`ParlayANN/algorithms/vamana/`. Output is streamed so the pass progress is visible.

In [ ]:
import subprocess


def run_vamana(binary, base_path, graph_out, R, L, alpha, extra_args=()):
    """Build a Vamana index, streaming the binary's output."""
    binary = Path(binary).resolve()
    if not binary.exists():
        raise FileNotFoundError(
            f"{binary} not found - run `make` in ParlayANN/algorithms/vamana/ first")

    cmd = [
        str(binary),
        "-R", str(R), "-L", str(L), "-alpha", str(alpha),
        "-data_type", "float", "-dist_func", "Euclidian",
        "-base_path", str(Path(base_path).resolve()),
        "-graph_outfile", str(Path(graph_out).resolve()),
        *extra_args,
    ]
    print(" ".join(cmd), "\n")

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"neighbors exited with status {proc.returncode}")
    return graph_out

In [ ]:
run_vamana(VAMANA_BIN, BASE_FBIN, GRAPH_PATH, R, L, ALPHA, extra_args=["-verbose"])

## 3. Coverage stats

ParlayANN's graph file stores only edges, so the uncovered counts are recomputed with the
same alpha-reachability rule as `distributed_robust_prune.py`: waypoint `w` covers `p`
when `d(w, p) * alpha < d(s, p)`. Distances are squared euclidean, so the test scales by
`alpha^2` and `p` stays uncovered while `d2(s, p) <= d2(w, p) * alpha_sq`.

Each edge records the uncovered count *after* it takes effect, matching the adj-list
writer. Edges are replayed in the order ParlayANN stored them.

Distances are accumulated in `DTYPE` (float64 by default). The covering test is an
inequality, so points sitting exactly on the `d(s, p) == d(w, p)` boundary - common at
`alpha = 1` - can flip between a blocked matmul and a per-vector pass when rounding in
float32. float64 removes that ambiguity and keeps counts reproducible.

The write is resumable: rerunning with the same `out_path` scans the lines already
there, drops a trailing line left half-written by an interrupt, and continues from
the first node still missing. Pass `resume=False` to start over from scratch.


In [ ]:
def read_csr(path):
    """Read a ParlayANN graph file into (list of neighbor arrays, max_deg)."""
    with open(path, "rb") as f:
        n, max_deg = np.fromfile(f, dtype=np.uint32, count=2)
        sizes = np.fromfile(f, dtype=np.uint32, count=int(n))
        edges = np.fromfile(f, dtype=np.uint32)
    offsets = np.concatenate([[0], np.cumsum(sizes, dtype=np.int64)])
    return [edges[offsets[i]:offsets[i + 1]] for i in range(int(n))], int(max_deg)

In [ ]:
# Use the GPU when CuPy is importable and a device is actually present.
try:
    import cupy as cp
    cp.cuda.runtime.getDeviceCount()
    xp = cp
    print("using GPU:", cp.cuda.runtime.getDeviceProperties(0)["name"].decode())
except Exception as exc:
    xp = np
    print(f"using CPU ({type(exc).__name__}: {exc})")

on_gpu = xp is not np

In [ ]:
def neighborhood_with_uncov(source, neighbors, V, norms, alpha_sq, chunk=32):
    """Replay one node's edges, recording the uncovered count after each edge.

    Distances to a block of waypoints come from one matmul, and the block is
    restricted to the rows still uncovered - that set shrinks quickly, so later
    blocks do far less work than a full pass over all points.
    """
    p = V[source]
    d_source = norms - 2.0 * (V @ p) + p @ p
    uncov = xp.arange(len(V), dtype=xp.int32)
    uncov = uncov[uncov != source]
    d_uncov = d_source[uncov]

    neighbors = xp.asarray(np.asarray(neighbors, dtype=np.int64))
    out = []
    for start in range(0, len(neighbors), chunk):
        block = neighbors[start:start + chunk]
        if len(uncov) == 0:                       # nothing left to cover
            out.extend((int(w), 0) for w in block.tolist())
            continue

        # (|uncov|, dim) @ (dim, block) -> squared distances for the whole block
        D = (norms[uncov][:, None]
             - 2.0 * (V[uncov] @ V[block].T)
             + norms[block][None, :])

        for t, w in enumerate(block.tolist()):
            keep = d_uncov <= D[:, t] * alpha_sq
            uncov, d_uncov, D = uncov[keep], d_uncov[keep], D[keep]
            # Drop the waypoint itself: for duplicate points d(w, w) = 0 makes the
            # test true, so it would otherwise never leave the uncovered set.
            m = uncov != w
            uncov, d_uncov, D = uncov[m], d_uncov[m], D[m]
            out.append((int(w), int(len(uncov))))
    return out

In [ ]:
import ast
import os


def completed_nodes(path):
    """Count the leading nodes already written to `path`, repairing a partial tail.

    An interrupted run can leave the last line truncated, so each line is parsed
    and checked against the node index it should carry. The file is truncated at
    the last line that passes, making the return value the next node to compute.
    """
    if not os.path.exists(path):
        return 0

    good_bytes = 0
    done = 0
    with open(path, "r+") as f:
        for line in f:
            stripped = line.strip()
            if not stripped or not line.endswith("\n"):
                break                              # blank or truncated tail
            try:
                index, payload = stripped.split(" ", 1)
                if int(index) != done:
                    break                          # out of order, stop here
                ast.literal_eval(payload)
            except (ValueError, SyntaxError):
                break
            good_bytes += len(line.encode())
            done += 1
        f.truncate(good_bytes)                     # drop the unusable tail
    return done


In [ ]:
def write_adj_list(graph_path, vectors_path, out_path, alpha=1.0, limit=None,
                   chunk=32, report_every=1000, dtype="float64", resume=True):
    """Recompute coverage for every node and write the adj-list file.

    With `resume=True` an existing `out_path` is continued rather than rewritten:
    the nodes already in the file are kept and the loop picks up at the first one
    missing. Each line is flushed as it is written so an interrupt costs at most
    the node in flight.
    """
    if alpha < 1.0:
        raise ValueError(f"alpha must be >= 1, got {alpha}")
    alpha_sq = alpha ** 2

    graph, _ = read_csr(graph_path)
    total = min(len(graph), limit) if limit else len(graph)

    start_at = completed_nodes(out_path) if resume else 0
    if start_at >= total:
        print(f"{out_path} already has {start_at} nodes, nothing to do")
        return out_path
    if start_at:
        print(f"resuming {out_path} at node {start_at}/{total}", flush=True)

    V = xp.asarray(read_fbin(vectors_path), dtype=dtype)
    norms = xp.einsum("ij,ij->i", V, V)

    mode = "a" if start_at else "w"
    with open(out_path, mode) as out:
        for i in range(start_at, total):
            nbrs = neighborhood_with_uncov(i, graph[i], V, norms, alpha_sq, chunk)
            out.write(f"{i} {nbrs}\n")
            out.flush()                    # a kill after this point loses nothing
            if (i + 1) % report_every == 0:
                print(f"{i + 1}/{total} nodes", flush=True)

    print(f"wrote {out_path} ({total} nodes)")
    return out_path


In [ ]:
write_adj_list(GRAPH_PATH, BASE_FBIN, ADJ_PATH,
               alpha=COVERAGE_ALPHA, limit=LIMIT, dtype=DTYPE)

## Check the output

Parse the result the way the analysis scripts do and confirm `uncov` is non-increasing.

In [ ]:
import ast

rows = []
with open(ADJ_PATH) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        source = line if line.startswith("[") else line[line.index(" ") + 1:]
        rows.append(ast.literal_eval(source))

n_nodes = len(read_fbin(BASE_FBIN))
for i, nb in enumerate(rows):
    uncovs = [u for _, u in nb]
    assert all(a >= b for a, b in zip(uncovs, uncovs[1:])), f"node {i} not monotone"
    assert all(0 <= n_nodes - u <= n_nodes for u in uncovs), f"node {i} out of range"

degrees = [len(nb) for nb in rows]
print(f"parsed {len(rows)} neighborhoods; uncov non-increasing and in range")
print(f"degree: mean {np.mean(degrees):.1f}, max {max(degrees)}")
print("cov_by_edge (node 0, first 8):", [n_nodes - u for _, u in rows[0]][:8])